In [ ]:
!pip install gensim
!python -m spacy download pt_core_news_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 29.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 71.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
# LAB 01: Troca do Algoritmo de Classificação (ML)

# BLOCO 1: PREPARAÇÃO

import re
import numpy as np
import pandas as pd
import spacy
import gensim.downloader as api
import gradio as gr

# LAB01:
# Importação da Árvore de Decisão
from sklearn.tree import DecisionTreeClassifier


# CARREGAMENTO

print("Carregando modelo Spacy...")

nlp = spacy.load(
    "pt_core_news_sm"
)

print("Carregando GloVe...")

word_vectors = api.load(
    "glove-wiki-gigaword-50"
)


# DATASET

dados_imobiliaria = [

    # --------------------------------------------------------
    # comprar_imovel
    # --------------------------------------------------------

    (
        "Quero comprar um apartamento de 3 quartos com varanda",
        "comprar_imovel"
    ),

    (
        "Gostaria de ver casas à venda no centro da cidade",
        "comprar_imovel"
    ),

    (
        "Qual o preço médio para compra de cobertura com piscina?",
        "comprar_imovel"
    ),

    (
        "Procuro imóvel residencial para comprar com financiamento",
        "comprar_imovel"
    ),

    (
        "Vocês têm sobrados à venda na zona sul?",
        "comprar_imovel"
    ),


    # --------------------------------------------------------
    # alugar_imovel
    # --------------------------------------------------------

    (
        "Procurando kitnet para alugar perto da faculdade",
        "alugar_imovel"
    ),

    (
        "Qual o valor do aluguel deste apartamento de 2 dormitórios?",
        "alugar_imovel"
    ),

    (
        "Quero alugar um galpão comercial para minha empresa",
        "alugar_imovel"
    ),

    (
        "Quais imóveis estão disponíveis para locação imediata?",
        "alugar_imovel"
    ),

    (
        "Preciso de uma casa para alugar que aceite animais",
        "alugar_imovel"
    ),


    # --------------------------------------------------------
    # suporte_manutencao
    # --------------------------------------------------------

    (
        "O chuveiro do apartamento alugado queimou, como pedir conserto?",
        "suporte_manutencao"
    ),

    (
        "Muro da casa está com infiltração e vazamento de água",
        "suporte_manutencao"
    ),

    (
        "Preciso do contato do encanador para reparo na cozinha",
        "suporte_manutencao"
    ),

    (
        "A porta da varanda quebrou, quem faz a manutenção?",
        "suporte_manutencao"
    ),

    (
        "Vazamento no teto do banheiro precisa de reparo urgente",
        "suporte_manutencao"
    ),


    # --------------------------------------------------------
    # 2via_boleto_contrato
    # --------------------------------------------------------

    (
        "Como faço para baixar a segunda via do boleto do aluguel?",
        "2via_boleto_contrato"
    ),

    (
        "Não recebi o boleto deste mês para pagamento",
        "2via_boleto_contrato"
    ),

    (
        "Preciso do informe de rendimentos e cópia do contrato",
        "2via_boleto_contrato"
    ),

    (
        "Onde pego o boleto atualizado com o valor do condomínio?",
        "2via_boleto_contrato"
    ),

    (
        "Quero solicitar a segunda via do recibo de pagamento",
        "2via_boleto_contrato"
    )
]


df = pd.DataFrame(
    dados_imobiliaria,
    columns=[
        "mensagem",
        "intencao"
    ]
)

print(
    f"Dataset carregado com {len(df)} mensagens "
    f"e {df['intencao'].nunique()} intenções."
)


# BLOCO 2: PRÉ-PROCESSAMENTO

def preprocessar_texto(texto: str) -> str:

    texto_limpo = texto.lower()

    texto_limpo = re.sub(
        r'[^a-záàâãéèêíïóôõöúçñ\s]',
        '',
        texto_limpo
    )

    doc = nlp(texto_limpo)

    tokens = [
        token.lemma_
        for token in doc
        if not token.is_stop
        and not token.is_space
        and len(token.text) > 1
    ]

    return " ".join(tokens)


def extrair_sentence_embedding(
    texto_limpo: str,
    modelo_emb
) -> np.ndarray:

    palavras = texto_limpo.split()

    vetores = [
        modelo_emb[p]
        for p in palavras
        if p in modelo_emb
    ]

    if len(vetores) == 0:

        return np.zeros(
            modelo_emb.vector_size
        )

    return np.mean(
        vetores,
        axis=0
    )


# VETORIZAÇÃO

df["mensagem_limpa"] = df[
    "mensagem"
].apply(
    preprocessar_texto
)

X_densos = np.array([
    extrair_sentence_embedding(
        txt,
        word_vectors
    )
    for txt in df["mensagem_limpa"]
])

y = df["intencao"].values

print(
    "Formato da matriz X:",
    X_densos.shape
)

print(
    "Formato do vetor y:",
    y.shape
)


# BLOCO 3: MODELO

# LAB01:
# Substituímos LogisticRegression
# por DecisionTreeClassifier.

modelo_nlu = DecisionTreeClassifier(
    random_state=42
)

modelo_nlu.fit(
    X_densos,
    y
)

print("\nÁrvore de Decisão treinada!")

print(
    "Classes aprendidas pelo modelo:"
)

print(
    modelo_nlu.classes_
)


# RESPOSTAS PADRÃO

RESPOSTAS_PADRAO = {

    "comprar_imovel": (
        "**Atendimento de Vendas:** Ficamos felizes "
        "com seu interesse! "
        "Você pode conferir nosso catálogo de imóveis "
        "à venda em www.imobiliaria.com/vendas."
    ),

    "alugar_imovel": (
        "**Atendimento de Locação:** Temos ótimas "
        "opções disponíveis! "
        "Acesse www.imobiliaria.com/aluguel para "
        "filtrar por região e valor."
    ),

    "suporte_manutencao": (
        "**Suporte e Manutenção:** Sentimos muito "
        "pelo inconveniente. "
        "Abra um chamado em "
        "www.imobiliaria.com/manutencao."
    ),

    "2via_boleto_contrato": (
        "**Financeiro e Contratos:** Acesse a Área "
        "do Cliente em "
        "www.imobiliaria.com/cliente para baixar "
        "sua 2ª via."
    )
}


# BLOCO 4: MOTOR DE INFERÊNCIA

LIMIAR_CONFIANCA = 0.50


def processar_atendimento_sac(
    mensagem_usuario: str
):

    if (
        not mensagem_usuario
        or not mensagem_usuario.strip()
    ):

        return (
            "N/A",
            "0.0%",
            "Aguardando mensagem...",
            "Aguardando entrada do usuário..."
        )


    # --------------------------------------------------------
    # 1. Pré-processamento
    # --------------------------------------------------------

    msg_limpa = preprocessar_texto(
        mensagem_usuario
    )


    # --------------------------------------------------------
    # 2. Vetorização
    # --------------------------------------------------------

    vetor_input = extrair_sentence_embedding(
        msg_limpa,
        word_vectors
    ).reshape(1, -1)


    # --------------------------------------------------------
    # 3. Predição
    # --------------------------------------------------------

    probabilidades = modelo_nlu.predict_proba(
        vetor_input
    )[0]

    idx_maior_prob = np.argmax(
        probabilidades
    )

    confianca = probabilidades[
        idx_maior_prob
    ]

    intencao_detectada = (
        modelo_nlu.classes_[
            idx_maior_prob
        ]
    )


    percentual_confianca = (
        f"{confianca * 100:.1f}%"
    )


    # --------------------------------------------------------
    # 4. Fallback
    # --------------------------------------------------------

    if confianca >= LIMIAR_CONFIANCA:

        classificacao_status = (
            f"IDENTIFICADO ({intencao_detectada})"
        )

        texto_resposta = RESPOSTAS_PADRAO[
            intencao_detectada
        ]

    else:

        classificacao_status = (
            "UNCERTAIN (Fallback Acionado)"
        )

        texto_resposta = (
            "Desculpe, não consegui compreender "
            "com clareza a sua solicitação. "
            "Estou transferindo agora mesmo sua "
            "conversa para um de nossos atendentes."
        )


    # --------------------------------------------------------
    # 5. Card de resposta
    # --------------------------------------------------------

    card_resposta = f"""
    <div style="
        background-color:#f0f4f9;
        border-left:5px solid #2b5c8f;
        padding:15px;
        border-radius:8px;
        margin-top:10px;
    ">

        <h4 style="
            margin:0 0 8px 0;
            color:#2b5c8f;
        ">
            Resposta Automática do SAC
        </h4>

        <p style="
            margin:0;
            font-size:15px;
            color:#1a1a1a;
        ">
            {texto_resposta}
        </p>

    </div>
    """


    return (
        intencao_detectada,
        percentual_confianca,
        classificacao_status,
        card_resposta
    )


# BLOCO 5: GRADIO

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="SAC Imobiliário - LAB01"
) as app:

    gr.Markdown(
        """
        # SAC Imobiliário — LAB01
        ### Classificação utilizando Árvore de Decisão
        """
    )


    with gr.Row():

        # ----------------------------------------------------
        # COLUNA ESQUERDA
        # ----------------------------------------------------

        with gr.Column(scale=1):

            gr.Markdown(
                "### Mensagem do Cliente"
            )

            input_texto = gr.Textbox(
                lines=4,
                placeholder=(
                    "Ex: Preciso da segunda via "
                    "do boleto de aluguel..."
                ),
                label="Digite sua necessidade"
            )

            btn_processar = gr.Button(
                "Processar Mensagem",
                variant="primary",
                size="lg"
            )


            gr.Examples(
                examples=[
                    [
                        "Preciso de suporte técnico "
                        "para consertar vazamento."
                    ],
                    [
                        "Quero ver apartamentos "
                        "à venda na zona sul."
                    ],
                    [
                        "Como faço para alugar "
                        "um galpão comercial?"
                    ],
                    [
                        "Gostaria de baixar "
                        "o boleto do condomínio."
                    ],
                    [
                        "Vocês vendem terreno "
                        "na Lua ou em Marte?"
                    ]
                ],
                inputs=input_texto
            )


        # ----------------------------------------------------
        # COLUNA DIREITA
        # ----------------------------------------------------

        with gr.Column(scale=1):

            gr.Markdown(
                "### Painel de Diagnóstico"
            )

            with gr.Row():

                out_intencao = gr.Textbox(
                    label="Intenção",
                    scale=2,
                    interactive=False
                )

                out_confianca = gr.Textbox(
                    label="Confiança",
                    scale=1,
                    interactive=False
                )


            out_status = gr.Textbox(
                label="Status da Decisão",
                interactive=False
            )


            out_resposta = gr.HTML(
                value=(
                    "<div style='padding:15px; color:#888;'>"
                    "Aguardando envio de mensagem..."
                    "</div>"
                ),
                label="Resposta da Imobiliária"
            )


    # --------------------------------------------------------
    # AÇÃO DO BOTÃO
    # --------------------------------------------------------

    btn_processar.click(
        fn=processar_atendimento_sac,
        inputs=[input_texto],
        outputs=[
            out_intencao,
            out_confianca,
            out_status,
            out_resposta
        ]
    )


# EXECUÇÃO

app.launch(
    debug=True,
    share=True
)


Carregando modelo Spacy...
Carregando GloVe...
Dataset carregado com 20 mensagens e 4 intenções.
Formato da matriz X: (20, 50)
Formato do vetor y: (20,)

Árvore de Decisão treinada!
Classes aprendidas pelo modelo:
['2via_boleto_contrato' 'alugar_imovel' 'comprar_imovel'
 'suporte_manutencao']


/tmp/ipykernel_3987/2808976701.py:449: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f16e53bb6189705786.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f16e53bb6189705786.gradio.live
